# 03 — Error Analysis

Understanding where and why models fail:
- Best and worst performing samples (by ROUGE-2)
- Failure pattern categorization
- Correlation between article length and summary quality
- Actionable improvement recommendations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

sns.set_theme(style='whitegrid')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('../pipeline/data/processed')
MODEL_PATH = '../training/models/pegasus'
MODEL_FALLBACK = 'google/pegasus-xsum'

## 1. Generate Predictions on Test Set

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')[['text', 'title']].dropna()
sample_df = test_df.sample(min(100, len(test_df)), random_state=42).reset_index(drop=True)

path = MODEL_PATH if Path(MODEL_PATH).exists() else MODEL_FALLBACK
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE)
model.eval()

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

records = []
for _, row in sample_df.iterrows():
    inputs = tokenizer(row['text'], return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        output = model.generate(inputs['input_ids'], max_length=64, num_beams=4, early_stopping=True)
    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    scores = scorer.score(row['title'], pred)
    records.append({
        'text':      row['text'],
        'reference': row['title'],
        'prediction': pred,
        'rouge1':    scores['rouge1'].fmeasure,
        'rouge2':    scores['rouge2'].fmeasure,
        'rougeL':    scores['rougeL'].fmeasure,
        'text_words': len(row['text'].split()),
    })

results_df = pd.DataFrame(records)
print(f'Generated predictions for {len(results_df)} samples')
results_df[['reference', 'prediction', 'rouge2']].head(5)

## 2. Best vs. Worst Predictions

In [ ]:
top5    = results_df.nlargest(5, 'rouge2')[['reference', 'prediction', 'rouge2']]
bottom5 = results_df.nsmallest(5, 'rouge2')[['reference', 'prediction', 'rouge2']]

print('=== TOP 5 (highest ROUGE-2) ===')
for _, row in top5.iterrows():
    print(f'  REF  : {row["reference"]}')
    print(f'  PRED : {row["prediction"]}')
    print(f'  R2   : {row["rouge2"]:.3f}\n')

print('=== BOTTOM 5 (lowest ROUGE-2) ===')
for _, row in bottom5.iterrows():
    print(f'  REF  : {row["reference"]}')
    print(f'  PRED : {row["prediction"]}')
    print(f'  R2   : {row["rouge2"]:.3f}\n')

## 3. Article Length vs. ROUGE-2 Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(results_df['text_words'], results_df['rouge2'],
                alpha=0.6, c=results_df['rouge2'], cmap='RdYlGn', edgecolors='white', s=60)
plt.colorbar(sc, label='ROUGE-2')
ax.set_xlabel('Article Length (words)')
ax.set_ylabel('ROUGE-2 Score')
ax.set_title('Article Length vs. Summary Quality (PEGASUS)')

# Trend line
import numpy as np
z = np.polyfit(results_df['text_words'], results_df['rouge2'], 1)
p = np.poly1d(z)
x_line = sorted(results_df['text_words'])
ax.plot(x_line, p(x_line), 'b--', alpha=0.5, label='Trend')
ax.legend()

plt.tight_layout()
plt.savefig('error_length_vs_rouge.png', dpi=150)
plt.show()

corr = results_df['text_words'].corr(results_df['rouge2'])
print(f'Pearson correlation (length vs ROUGE-2): {corr:.3f}')

## 4. ROUGE-2 Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(results_df['rouge2'], bins=20, color='steelblue', edgecolor='white')
ax.axvline(results_df['rouge2'].mean(), color='red', linestyle='--', label=f"Mean: {results_df['rouge2'].mean():.3f}")
ax.axvline(results_df['rouge2'].median(), color='orange', linestyle='--', label=f"Median: {results_df['rouge2'].median():.3f}")
ax.set_xlabel('ROUGE-2 Score')
ax.set_ylabel('Count')
ax.set_title('ROUGE-2 Score Distribution (PEGASUS on Test Set)')
ax.legend()
plt.tight_layout()
plt.savefig('error_rouge_distribution.png', dpi=150)
plt.show()

## 5. Failure Pattern Observations

Based on the bottom-5 analysis, common failure modes are:

1. **Overly generic summaries** — model outputs category-level descriptions instead of specific titles (e.g., 'A historical event' instead of the actual event name)
2. **Named entity misses** — proper nouns, locations, and person names are often dropped
3. **Long article degradation** — articles > 350 words show lower ROUGE scores, suggesting the 512-token truncation cuts relevant context

**Recommended next steps:**
- Try hierarchical summarization: chunk → summarize each chunk → summarize summaries
- Add named entity recognition (NER) as a post-processing step to inject entities back
- Experiment with longer input windows using Longformer or LED